In [ ]:
import os
import json
import glob
import zipfile
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Colab mac dinh lam viec trong /content.
# Doi MOUNT_DRIVE = True neu muon luu dataset vao Google Drive.
MOUNT_DRIVE = False

if IN_COLAB and MOUNT_DRIVE:
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/COde')
else:
    BASE_DIR = Path('/content/COde') if IN_COLAB else Path.cwd() / 'COde'

BASE_DIR.mkdir(parents=True, exist_ok=True)

print('Python OK')
print('Current dir:', os.getcwd())
print('Running in Colab:', IN_COLAB)
print('Dataset/work dir:', BASE_DIR)


In [ ]:
# Colab da co san pandas, pillow, matplotlib. Khong dung -U de tranh conflict.
!pip -q install huggingface_hub datasets pyarrow seaborn tqdm


In [ ]:
from huggingface_hub import snapshot_download

REPO_ID = 'zirak-ai/COde'

DATA_DIR = snapshot_download(
    repo_id=REPO_ID,
    repo_type='dataset',
    local_dir=str(BASE_DIR),
)

print('Dataset downloaded to:')
print(DATA_DIR)


# Extract

In [ ]:
from pathlib import Path
import zipfile
import os

root = BASE_DIR
zip_path = root / 'COde-Dataset.zip'
extract_dir = root / 'dataset'

extract_dir.mkdir(exist_ok=True)

print('ZIP:', zip_path)
print('Size:', zip_path.stat().st_size / 1024**2, 'MB')

with zipfile.ZipFile(zip_path, 'r') as z:
    print('Number of files in ZIP:', len(z.namelist()))

    # Xem truoc 50 path
    for name in z.namelist()[:50]:
        print(name)

    z.extractall(extract_dir)

print('\nExtracted to:', extract_dir)


In [ ]:
files = []

for p in extract_dir.rglob("*"):
    if p.is_file():
        files.append({
            "path": str(p.relative_to(extract_dir)),
            "size_MB": round(p.stat().st_size / 1024**2, 3),
            "suffix": p.suffix.lower()
        })

file_df = (
    __import__("pandas")
    .DataFrame(files)
    .sort_values(["suffix", "path"])
    .reset_index(drop=True)
)

display(file_df.head(200))

In [ ]:
display(
    file_df.groupby("suffix")
    .agg(
        n_files=("path", "count"),
        total_MB=("size_MB", "sum")
    )
    .sort_values("total_MB", ascending=False)
)

In [ ]:
tabular_files = [
    p for p in extract_dir.rglob("*")
    if p.is_file()
    and p.suffix.lower() in [
        ".csv", ".json", ".jsonl", ".parquet"
    ]
]

print("Tabular files:")

for p in tabular_files:
    print(
        f"{p.relative_to(extract_dir)} | "
        f"{p.stat().st_size / 1024**2:.2f} MB"
    )

# View some samples

In [ ]:
import pandas as pd
import json

for p in tabular_files:

    print("\n" + "="*100)
    print(p.relative_to(extract_dir))

    try:

        if p.suffix.lower() == ".csv":
            tmp = pd.read_csv(p, nrows=5)

        elif p.suffix.lower() == ".parquet":
            tmp = pd.read_parquet(p).head()

        elif p.suffix.lower() == ".jsonl":
            rows = []

            with open(p, "r", encoding="utf-8") as f:
                for i, line in enumerate(f):
                    if i >= 5:
                        break

                    if line.strip():
                        rows.append(json.loads(line))

            tmp = pd.DataFrame(rows)

        elif p.suffix.lower() == ".json":
            with open(p, "r", encoding="utf-8") as f:
                obj = json.load(f)

            if isinstance(obj, list):
                tmp = pd.DataFrame(obj[:5])

            elif isinstance(obj, dict):
                tmp = pd.DataFrame([obj])

            else:
                tmp = pd.DataFrame({"raw": [obj]})

        print("Shape/sample:", tmp.shape)
        print("Columns:")
        print(list(tmp.columns))

        display(tmp.head())

    except Exception as e:
        print("ERROR:", e)